# Alchemy Ingredients — Membership Testing & List Comparison

**Topic:** comparing two lists with the `in` keyword, building a percentage, and collecting the difference.

This notebook is your own copy to run, break, and re-run. Every code cell has real output underneath it (I actually ran this, I'm not describing what it "should" do).

## 1. The concept

You have two lists of strings:

- `recipe` — everything the potion needs
- `inventory` — everything the character actually has

You need two things back:
1. **What fraction of the recipe is covered** — a percentage.
2. **What's missing** — a new list, in the recipe's original order.

The tool that makes this easy is the `in` keyword — Python's **membership test operator**. `item in some_collection` asks "does this collection contain this item?" and returns `True`/`False`. `item not in some_collection` is the negation.

There's a second, quieter concept hiding here too: **sets**. A `set` is an unordered collection with no duplicates. You'll see why that matters in a second.

## 2. Why it matters

**ELI5 first:** Imagine checking off a shopping list against what's already in your fridge. For each thing on the list, you look in the fridge and say "got it" or "need it." That's exactly what `in` does — it's the "look in the fridge" step.

**Now the real explanation:** the *type* of collection you check membership against changes performance, not just style.

- `item in some_list` — Python walks the list from the start until it finds a match (or reaches the end). Checking one item costs up to *N* steps, where N is the list's length.
- `item in some_set` — Python uses hashing to jump almost straight to where the item would be. Checking one item costs roughly 1 step, regardless of the set's size.

If `recipe` has *R* ingredients and `inventory` has *I* ingredients, checking membership against the raw `inventory` list costs about `R × I` operations in the worst case. Checking against `set(inventory)` costs about `R + I` (one pass to build the set, then O(1) per check). For two short lists in a game, this difference is invisible. For a real inventory system with thousands of items, it's the difference between instant and noticeably slow.

**Beginner mistake to avoid:** doing `if item in inventory` inside a loop, over and over, without first converting `inventory` to a set. It still gives the *correct* answer — it's just needlessly slow once the lists get big.

In [1]:
# --- Simple example: membership testing on its own ---

fridge = ["eggs", "milk", "butter"]

print("eggs" in fridge)        # True  -> present
print("flour" in fridge)       # False -> absent
print("flour" not in fridge)   # True  -> the negated form, reads naturally in an if-statement

# Converting to a set changes nothing about the RESULT, only the speed of the check
fridge_set = set(fridge)
print("eggs" in fridge_set)

True
False
True
True


## 3. Real-world example

Think of a warehouse system checking whether an order can be fulfilled from stock. `recipe` becomes the order's line items, `inventory` becomes the warehouse's stock list. The same shape of problem — "what fraction of this order can I fill right now, and what do I need to restock?" — shows up in e-commerce, manufacturing bill-of-materials checks, and, here, a fantasy game's crafting system.

That's worth noticing: `check_ingredient_match` isn't really an "alchemy" function. It's a general **set-comparison / coverage-percentage** function wearing a fantasy costume. You'll meet this exact pattern again under a different name.

## 4. Small challenge (try before scrolling to the solution)

Before you look at the solution cell below, try writing the function yourself. Some questions to think through first:

1. How do you turn a *count* of missing items into a *percentage of items you have*?
2. If you loop over `recipe` and check `if ingredient in inventory`, does that preserve the recipe's original order in your `missing` list? (Yes — this matters, and it's why a plain loop or list comprehension beats `set(recipe) - set(inventory)` here, since set subtraction doesn't guarantee order.)
3. What should happen if `recipe` is empty? (Edge case worth thinking about, even though the assignment doesn't test it — dividing by `len(recipe)` will crash if it's 0.)

Write your attempt in the empty cell below, then compare with the solution.

In [2]:
# Your attempt here
def check_ingredient_match(recipe, inventory):
    pass


## 5. Solution, with reasoning in the comments

In [3]:
def check_ingredient_match(recipe, inventory):
    """
    Compare a required ingredient list against what's on hand.

    Args:
        recipe (list[str]): every ingredient required (no duplicates).
        inventory (list[str]): every ingredient currently held.

    Returns:
        tuple[float, list[str]]: (percentage of recipe covered, missing ingredients)
    """
    # Turn inventory into a set once, up front. This is the O(1)-membership-check
    # trick from section 2 -- we pay a small one-time cost to build the set,
    # then every "in" check below is fast regardless of inventory size.
    inventory_set = set(inventory)

    # Walk the recipe IN ITS ORIGINAL ORDER and keep whatever isn't in inventory.
    # Using a list comprehension over `recipe` (not a set difference) is what
    # preserves that order -- sets have no guaranteed order, so
    # `set(recipe) - inventory_set` could hand back "Troll Tusk" before
    # "Unicorn Hair" even though Unicorn Hair came first in the recipe.
    missing = [ingredient for ingredient in recipe if ingredient not in inventory_set]

    have_count = len(recipe) - len(missing)
    percentage = (have_count / len(recipe)) * 100

    # round() gets us to 75.0 rather than 74.999999998 style float noise.
    # If you need it displayed as exactly "75.00" (two decimal PLACES, not just
    # two decimal digits of precision), that's a formatting job for the caller:
    #   f"{percentage:.2f}"
    return round(percentage, 2), missing


## 6. Verifying it — real output, not predicted output

Run the cell below yourself. This is the exact example from the assignment, plus a few edge cases worth checking.

In [4]:
recipe = ["Dragon Scale", "Unicorn Hair", "Phoenix Feather", "Troll Tusk"]
inventory = ["Dragon Scale", "Phoenix Feather", "Troll Tusk"]

percentage, missing_ingredients = check_ingredient_match(recipe, inventory)
print(percentage, missing_ingredients)
# Expected from the assignment: 75.0 ['Unicorn Hair']

# Edge cases
print(check_ingredient_match(["A", "B"], []))               # nothing in stock -> 0.0, ['A', 'B']
print(check_ingredient_match(["A", "B"], ["A", "B"]))        # fully stocked -> 100.0, []
print(check_ingredient_match(["A", "B"], ["A", "B", "C"]))   # extra inventory items are ignored -> 100.0, []

75.0 ['Unicorn Hair']
(0.0, ['A', 'B'])
(100.0, [])
(100.0, [])


## 7. Common beginner mistakes with this pattern

- **Checking membership against the raw list in a loop, on a huge inventory.** Correct, but slow at scale — convert to a `set` first (see section 2).
- **Using `set(recipe) - set(inventory)` for `missing`.** This gives the *right members* but not necessarily the *right order*, since Python sets don't preserve insertion order the way lists do.
- **Dividing by `len(recipe)` without checking it's non-zero.** An empty recipe list would raise `ZeroDivisionError`. Not required by this assignment, but worth flagging as the kind of edge case real code has to consider.
- **Confusing `round(x, 2)` with string formatting.** `round(75.0, 2)` is the float `75.0`, which Python prints as `75.0`, not `75.00`. If you need the *string* `"75.00"`, that's `f"{x:.2f}"` — rounding controls the value, formatting controls the display.

## 8. Alternative approaches (tradeoffs)

| Approach | Order preserved? | Speed on large inputs | Notes |
|---|---|---|---|
| List comprehension + `set(inventory)` (used above) | Yes | Fast (O(R + I)) | Best general-purpose choice |
| Plain loop, `if ingredient in inventory` (list, not set) | Yes | Slower (O(R × I)) | Fine for tiny lists, avoid for big ones |
| `set(recipe) - set(inventory)` | **No** | Fast | Simplest to write, but breaks the "missing in recipe order" requirement |

## 9. Terms introduced here

- **Membership test operator (`in` / `not in`)** — checks whether a value exists inside a collection.
- **Set** — an unordered collection of unique values; supports fast membership checks and set algebra (`-`, `&`, `|`).
- **List comprehension** — `[expr for item in iterable if condition]`, a compact way to build a filtered/transformed list.
- **Time complexity (informal)** — a rough sense of how an operation's cost grows with input size; here, O(1) per set-lookup vs O(n) per list-lookup.